# PYTORCH MASTERY — NOTEBOOK 14
## Chapters 79–85: Large-Scale Training → Production Project Structure → Testing PyTorch Code → End-to-End CV Project → End-to-End NLP Project → End-to-End LLM Project → End-to-End Recommendation Project

Continuing directly from Notebook 13. This notebook begins the course's run of complete, portfolio-quality projects.


### PYTORCH MASTERY PROGRESS
Current Chapter: 85 | Current Level: Engineering Discipline & Portfolio Projects

- [x] Chapters 1–78 *(Notebooks 1–13)*
- [x] Large-Scale Training
- [x] Production Project Structure
- [x] Testing PyTorch Code
- [x] **Project: End-to-End Computer Vision**
- [x] **Project: End-to-End NLP**
- [x] **Project: End-to-End LLM**
- [x] **Project: End-to-End Recommendation System**


---
# CHAPTER 79 — LARGE-SCALE TRAINING

## 1. Concept
This chapter consolidates every scale-related technique from Notebooks 12-13 into the engineering considerations genuinely required for training at serious scale (millions of samples, large models, multi-GPU) — fault tolerance, checkpointing cadence, and experiment reproducibility under real-world operational constraints.

## 2. Fault Tolerance — Training WILL Be Interrupted


In [ ]:
print("""
At large scale, training runs span HOURS to WEEKS -- over this duration, SOME
kind of interruption becomes a near-certainty: a node failure, a preemptible/
spot cloud instance being reclaimed (common and expected -- these instances are
genuinely CHEAPER specifically because the provider can reclaim them with
little notice), a scheduled maintenance window, or simply a bug discovered
mid-run requiring a restart.

THE ENGINEERING IMPLICATION: large-scale training must be DESIGNED for
interruption from the START, not treated as an unfortunate exception --
Chapter 72's checkpointing becomes not just good practice but an ABSOLUTE
REQUIREMENT, and checkpoint FREQUENCY becomes a genuine, deliberate tradeoff:

  - Checkpoint TOO rarely: a failure near the end of a long interval between
    checkpoints loses a LOT of (expensive) compute progress.
  - Checkpoint TOO frequently: the checkpointing operation itself (writing a
    multi-gigabyte file to disk/cloud storage) has a REAL time cost,
    directly competing with actual training time if done too often.

A COMMON, reasonable starting point: checkpoint every N steps/minutes such
that, in expectation, losing progress since the LAST checkpoint represents an
ACCEPTABLE (small, bounded) fraction of total training time -- genuinely
similar in spirit to choosing a database backup frequency based on acceptable
data-loss tolerance.
""")



At large scale, training runs span HOURS to WEEKS -- over this duration, SOME
kind of interruption becomes a near-certainty: a node failure, a preemptible/
spot cloud instance being reclaimed (common and expected -- these instances are
genuinely CHEAPER specifically because the provider can reclaim them with
little notice), a scheduled maintenance window, or simply a bug discovered
mid-run requiring a restart.

THE ENGINEERING IMPLICATION: large-scale training must be DESIGNED for
interruption from the START, not treated as an unfortunate exception --
Chapter 72's checkpointing becomes not just good practice but an ABSOLUTE
REQUIREMENT, and checkpoint FREQUENCY becomes a genuine, deliberate tradeoff:

  - Checkpoint TOO rarely: a failure near the end of a long interval between
    checkpoints loses a LOT of (expensive) compute progress.
  - Checkpoint TOO frequently: the checkpointing operation itself (writing a
    multi-gigabyte file to disk/cloud storage) has a REAL time cost,
    

## 3. Reproducibility at Scale — Beyond Chapter 12's Single-GPU Seeding


In [ ]:
print("""
Chapter 12's full reproducibility setup (seeding CPU, GPU, Python, NumPy,
cuDNN determinism) becomes MEANINGFULLY HARDER to guarantee at large,
distributed scale:

  - DistributedSampler's (Chapter 69) shuffling must be seeded CONSISTENTLY
    and CORRECTLY across every participating process.
  - The ORDER in which distributed gradient synchronization (the "all-reduce"
    operation, Chapter 69) combines floating-point values across MANY devices
    can itself introduce small numerical non-determinism, since floating-point
    addition is NOT perfectly associative -- summing the SAME values in a
    DIFFERENT order can produce a SLIGHTLY different result, and the exact
    order of a distributed reduction can vary run to run depending on network
    timing.
  - Mixed precision (Chapter 29) and cuDNN's autotuning (Chapter 12) both
    introduce additional sources of potential non-determinism at scale.

PRACTICAL GUIDANCE: for large-scale training, treat "exact bit-for-bit
reproducibility" as a GENUINELY difficult, sometimes not fully achievable goal
(directly extending Chapter 12's "identical seeds don't always guarantee
perfect reproducibility" lesson) -- instead, focus reproducibility efforts on
what's GENUINELY achievable and valuable: consistent, documented
HYPERPARAMETERS and data, STABLE aggregate final metrics across re-runs
(even if not bit-identical), and COMPREHENSIVE experiment tracking (Chapter 34)
so that "what configuration produced this result" is always answerable, even
when "the exact same bits" isn't guaranteed.
""")



Chapter 12's full reproducibility setup (seeding CPU, GPU, Python, NumPy,
cuDNN determinism) becomes MEANINGFULLY HARDER to guarantee at large,
distributed scale:

  - DistributedSampler's (Chapter 69) shuffling must be seeded CONSISTENTLY
    and CORRECTLY across every participating process.
  - The ORDER in which distributed gradient synchronization (the "all-reduce"
    operation, Chapter 69) combines floating-point values across MANY devices
    can itself introduce small numerical non-determinism, since floating-point
    addition is NOT perfectly associative -- summing the SAME values in a
    DIFFERENT order can produce a SLIGHTLY different result, and the exact
    order of a distributed reduction can vary run to run depending on network
    timing.
  - Mixed precision (Chapter 29) and cuDNN's autotuning (Chapter 12) both
    introduce additional sources of potential non-determinism at scale.

PRACTICAL GUIDANCE: for large-scale training, treat "exact bit-for-bit
reproducibili

### Common Mistakes
- Not implementing checkpointing until after a long-running training job has already failed and lost significant progress — a costly, avoidable lesson learned the hard way.
- Expecting bit-for-bit reproducibility from large-scale distributed training — an unrealistic standard that, when pursued too rigidly, can distract from the genuinely achievable and valuable reproducibility goals (consistent configuration, comparable aggregate results).
- Checkpointing too infrequently for the actual cost/risk profile of the training environment (e.g., using a conservative checkpoint interval appropriate for reliable dedicated hardware, while actually running on frequently-preempted spot instances).

### Interview Insight
**Q: Why does achieving bit-for-bit reproducibility become genuinely harder at large, multi-GPU distributed scale, even with every single-GPU reproducibility technique (Chapter 12) correctly applied?**
A: Distributed training introduces additional sources of non-determinism beyond single-GPU training: gradient synchronization across many devices (the "all-reduce" operation) combines floating-point values from different devices, and because floating-point addition isn't perfectly associative, the exact order in which these values get combined — which can itself vary run to run based on network timing and scheduling — can produce slightly different numerical results even with identical starting conditions and seeds. Combined with mixed precision's own potential sources of non-determinism and cuDNN's autotuning, achieving genuinely bit-for-bit identical results across separate large-scale distributed runs becomes a substantially harder, sometimes practically unachievable goal — the more realistic and valuable practice is to focus on consistent, well-documented configuration and comprehensive experiment tracking, accepting that final metrics may vary very slightly (though should remain stable in aggregate) across re-runs.

### Quiz (Chapter 79)
1. Why must large-scale training be designed for interruption from the start, rather than treating failures as rare exceptions?
2. What's the genuine tradeoff in choosing checkpoint frequency?
3. Why does distributed gradient synchronization introduce a source of non-determinism that single-GPU training doesn't have?


---
# CHAPTER 80 — PRODUCTION PROJECT STRUCTURE

## 1. Concept
Directly extending the Scikit-learn course's Chapter 78 project-structure discipline into PyTorch's specific needs — the standard professional layout for a real PyTorch project.

## 2. The Standard Layout


In [ ]:
project_structure = """
pytorch-project/
|
+-- data/
|   +-- raw/              -- never modified after initial collection
|   +-- processed/        -- cleaned/preprocessed, reproducible from raw/ + src/
|
+-- configs/               -- YAML/JSON hyperparameter configs -- NOT hardcoded in src/
|   +-- base_config.yaml
|   +-- experiment_v2.yaml
|
+-- notebooks/              -- exploration, numbered by workflow stage (Scikit-learn course Ch 89's discipline)
|   +-- 01_eda.ipynb
|   +-- 02_baseline.ipynb
|
+-- src/                     -- STABLE, TESTED, reusable code
|   +-- datasets/             -- Dataset/DataLoader construction (Chapters 24-26)
|   +-- models/                -- nn.Module definitions (Chapters 16-17, 35-51)
|   +-- training/               -- training loop logic (Chapter 23)
|   +-- evaluation/              -- metrics, evaluation functions
|   +-- inference/                -- prediction/serving logic (Chapters 62-63)
|   +-- utils/                     -- shared utilities (checkpointing Ch 72, seeding Ch 12)
|
+-- scripts/                        -- entry-point scripts (train.py, evaluate.py, export.py)
|
+-- checkpoints/                     -- persisted model artifacts + metadata (Chapter 72)
|
+-- tests/                            -- unit tests for src/ (Chapter 81)
|
+-- requirements.txt                   -- pinned dependency versions (Chapter 64's Docker lesson)
|
+-- Dockerfile                          -- Chapter 64
|
+-- README.md
"""
print(project_structure)



pytorch-project/
|
+-- data/
|   +-- raw/              -- never modified after initial collection
|   +-- processed/        -- cleaned/preprocessed, reproducible from raw/ + src/
|
+-- configs/               -- YAML/JSON hyperparameter configs -- NOT hardcoded in src/
|   +-- base_config.yaml
|   +-- experiment_v2.yaml
|
+-- notebooks/              -- exploration, numbered by workflow stage (Scikit-learn course Ch 89's discipline)
|   +-- 01_eda.ipynb
|   +-- 02_baseline.ipynb
|
+-- src/                     -- STABLE, TESTED, reusable code
|   +-- datasets/             -- Dataset/DataLoader construction (Chapters 24-26)
|   +-- models/                -- nn.Module definitions (Chapters 16-17, 35-51)
|   +-- training/               -- training loop logic (Chapter 23)
|   +-- evaluation/              -- metrics, evaluation functions
|   +-- inference/                -- prediction/serving logic (Chapters 62-63)
|   +-- utils/                     -- shared utilities (checkpointing Ch 72, s

### Why Configuration Files (Not Hardcoded Values) Matter at This Scale
Directly extending the Scikit-learn course's Chapter 89 "configuration block" discipline — for PyTorch projects specifically, a YAML/JSON config file (rather than even a single Python dict) is standard practice because it allows launching many different experiment configurations **without modifying any code at all** — genuinely essential for the kind of systematic hyperparameter exploration this course's models require, and for keeping a clear, versioned record of exactly which configuration produced which trained model (directly supporting the Scikit-learn course's Chapter 67 versioning discipline).

### Common Mistakes
- Keeping all project logic inside notebooks, never promoting stable, reusable logic into `src/` — exactly the Scikit-learn course's Chapter 78 lesson, now even more consequential given PyTorch projects' typically larger, more complex codebases (custom datasets, custom architectures, custom training loops).
- Hardcoding hyperparameters directly in training scripts rather than externalizing them into config files — makes systematic experimentation and reproducibility meaningfully harder.
- Not separating `models/` (architecture definitions) from `training/` (the training loop logic) — conflating these makes it harder to reuse the same architecture with a different training procedure, or vice versa.

### Interview Insight
**Q: Why do professional PyTorch projects typically externalize hyperparameters into YAML/JSON config files rather than hardcoding them in Python scripts?**
A: Config files let you launch many distinct experiment configurations (different learning rates, architectures, batch sizes) without touching any code at all — reducing the risk of accidentally introducing a code bug while merely trying to adjust a hyperparameter, and making it trivial to version-control and precisely compare exactly which configuration produced which trained model/result, directly supporting rigorous experiment tracking (Chapter 34) and model versioning (Scikit-learn course Chapter 67) disciplines. This separation of "what the code does" from "what specific values it's run with" is a fundamental software engineering principle that becomes especially valuable in ML specifically because systematic hyperparameter exploration (trying many configurations) is such a core, repeated part of the workflow.

### Quiz (Chapter 80)
1. What's the core principle distinguishing what belongs in `notebooks/` versus `src/`?
2. Why do PyTorch projects typically use external config files rather than hardcoded hyperparameters?
3. Why separate `models/` from `training/` in the project structure?


---
# CHAPTER 81 — TESTING PYTORCH CODE

## 1. Concept
Directly extending the Scikit-learn course's Chapter 79 testing discipline with PyTorch-specific test patterns: shape tests, gradient tests, and dataset/checkpoint tests.

## 2. A Genuine PyTorch Test Suite


In [ ]:
import unittest
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self, input_dim=10, output_dim=2):
        super().__init__()
        self.fc = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.fc(x)


class TestModelShapes(unittest.TestCase):
    """SHAPE tests -- the most common, highest-value PyTorch test category,
    directly catching the shape-mismatch bugs Chapter 33 extensively covered."""

    def setUp(self):
        self.model = SimpleModel(input_dim=10, output_dim=2)

    def test_output_shape_matches_expected(self):
        x = torch.randn(4, 10)
        output = self.model(x)
        self.assertEqual(output.shape, (4, 2))

    def test_handles_batch_size_one(self):
        """Models with BatchNorm (Chapter 17) can FAIL specifically at batch_size=1 --
        a genuinely worthwhile edge case to test explicitly."""
        x = torch.randn(1, 10)
        output = self.model(x)
        self.assertEqual(output.shape, (1, 2))

    def test_raises_on_wrong_input_dim(self):
        x = torch.randn(4, 20)   # WRONG feature dimension
        with self.assertRaises(RuntimeError):
            self.model(x)


class TestGradientFlow(unittest.TestCase):
    """GRADIENT tests -- confirming gradients actually flow to every parameter,
    directly catching Chapter 33's 'model not learning' failure category."""

    def test_all_parameters_receive_gradients(self):
        model = SimpleModel()
        x = torch.randn(4, 10)
        output = model(x)
        loss = output.sum()
        loss.backward()

        for name, param in model.named_parameters():
            self.assertIsNotNone(param.grad, f"{name} received NO gradient")
            self.assertFalse(torch.isnan(param.grad).any(), f"{name} has a NaN gradient")

    def test_no_gradient_in_eval_mode_with_no_grad(self):
        model = SimpleModel()
        model.eval()
        x = torch.randn(4, 10)
        with torch.no_grad():
            output = model(x)
        self.assertFalse(output.requires_grad)


loader = unittest.TestLoader()
suite = unittest.TestSuite()
suite.addTests(loader.loadTestsFromTestCase(TestModelShapes))
suite.addTests(loader.loadTestsFromTestCase(TestGradientFlow))
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f"\n{result.testsRun} tests run, {len(result.failures)} failures, {len(result.errors)} errors")


test_handles_batch_size_one (__main__.TestModelShapes.test_handles_batch_size_one)
Models with BatchNorm (Chapter 17) can FAIL specifically at batch_size=1 -- ... ok
test_output_shape_matches_expected (__main__.TestModelShapes.test_output_shape_matches_expected) ... ok
test_raises_on_wrong_input_dim (__main__.TestModelShapes.test_raises_on_wrong_input_dim) ... ok
test_all_parameters_receive_gradients (__main__.TestGradientFlow.test_all_parameters_receive_gradients) ... ok
test_no_gradient_in_eval_mode_with_no_grad (__main__.TestGradientFlow.test_no_gradient_in_eval_mode_with_no_grad) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.325s

OK



5 tests run, 0 failures, 0 errors


### Line-by-Line Explanation
- `test_handles_batch_size_one` — a genuinely important, PyTorch-specific edge case: `nn.BatchNorm1d` (Chapter 17) computes statistics **across the batch dimension**, and a batch of size 1 provides no meaningful variance to compute — this can raise an error or produce degenerate behavior depending on the exact configuration, making it a real, worthwhile thing to explicitly test for any model that might ever be called with a single sample (e.g., during certain inference scenarios).
- `test_all_parameters_receive_gradients` — directly automates Chapter 33's "model not learning" diagnostic checklist: iterating `named_parameters()` and confirming every single one received a non-`None`, non-`NaN` gradient after a backward pass catches a whole class of "forgot to register a layer correctly" or "accidentally detached part of the graph" bugs automatically, as a standing regression test rather than a one-off manual debugging session.

### Checkpoint Round-Trip Testing


In [ ]:
import tempfile
import os

class TestCheckpointRoundTrip(unittest.TestCase):
    """Directly the Scikit-learn course's joblib round-trip verification pattern
    (Chapter 66 there), now applied to a PyTorch checkpoint."""

    def test_save_and_load_produces_identical_output(self):
        model = SimpleModel()
        model.eval()
        x = torch.randn(4, 10)

        with torch.no_grad():
            original_output = model(x)

        with tempfile.TemporaryDirectory() as tmpdir:
            path = os.path.join(tmpdir, "test_checkpoint.pt")
            torch.save(model.state_dict(), path)

            new_model = SimpleModel()
            new_model.load_state_dict(torch.load(path))
            new_model.eval()

            with torch.no_grad():
                loaded_output = new_model(x)

        self.assertTrue(torch.equal(original_output, loaded_output))


suite2 = loader.loadTestsFromTestCase(TestCheckpointRoundTrip)
result2 = runner.run(suite2)
print(f"\n{result2.testsRun} tests run, {len(result2.failures)} failures")


test_save_and_load_produces_identical_output (__main__.TestCheckpointRoundTrip.test_save_and_load_produces_identical_output) ... ok

----------------------------------------------------------------------
Ran 1 test in 0.019s

OK



1 tests run, 0 failures


### Common Mistakes
- Only testing the "happy path" (normal batch sizes, expected input shapes) without testing genuine edge cases (`batch_size=1`, malformed input) — misses real, common production failure modes.
- Not testing gradient flow at all — a model can run without error (producing *some* output) while silently having a layer that never actually trains, exactly Chapter 51's `nn.ModuleList` lesson, catchable systematically via this kind of test.
- Not using `tempfile.TemporaryDirectory()` for checkpoint tests — leaves test artifacts scattered on disk; using a temporary directory that's automatically cleaned up is standard, tidy testing practice.

### Interview Insight
**Q: Why is testing that "every parameter receives a gradient after backward()" a genuinely valuable automated test, rather than something you'd only check manually while debugging?**
A: A model can run without raising any error — producing seemingly valid output — while silently having one or more layers that never actually participate in training, due to bugs like defining a layer inside `forward()` instead of `__init__()` (Chapter 16), using a plain Python list instead of `nn.ModuleList` for a collection of submodules (Chapter 51), or accidentally detaching part of the computational graph. These bugs are easy to introduce during refactoring and easy to miss during manual code review, since the model still "runs." An automated test that iterates every named parameter after a backward pass and asserts none have a `None` or `NaN` gradient directly catches this entire class of bug automatically, as a standing regression test that runs on every code change — turning a subtle, hard-to-notice correctness issue into an immediate, clear test failure.

### Quiz (Chapter 81)
1. Why specifically test `batch_size=1` for a model using BatchNorm?
2. What does the "all parameters receive gradients" test catch that simply running the model successfully would not?
3. Why use `tempfile.TemporaryDirectory()` for checkpoint round-trip tests?


---
# CHAPTER 82 — END-TO-END PROJECT: INDUSTRIAL DEFECT DETECTION

## Business Problem
A manufacturing company wants to automatically flag defective products from images captured on the production line, assembling every CV technique from Notebooks 7-8 into one complete, portfolio-quality project.

## The Complete Workflow


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms

# === 1. DATA PIPELINE (Chapter 37) ===
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),            # SMALL rotations -- defects can appear at any orientation
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# === 2. MODEL: transfer learning from ResNet (Chapter 38-39) ===
num_classes = 2   # "defective" vs "acceptable"
model = models.resnet34(weights="IMAGENET1K_V1")
for param in model.parameters():
    param.requires_grad = False
model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))

# === 3. LOSS: class_weight for imbalance (defects are RARE in a healthy production line --
#     directly the Scikit-learn course's Chapter 45 imbalanced-data lesson, now in PyTorch) ===
class_weights = torch.tensor([1.0, 8.0])   # defective class weighted MUCH more heavily --
                                              # a FAR more common imbalance ratio in real QA data
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

print(f"Model ready. Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("Class weights applied directly inside CrossEntropyLoss -- the PyTorch-native")
print("equivalent of Scikit-learn's class_weight='balanced' (Ch 45 there).")


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 105MB/s]


Model ready. Trainable parameters: 1,026
Class weights applied directly inside CrossEntropyLoss -- the PyTorch-native
equivalent of Scikit-learn's class_weight='balanced' (Ch 45 there).


In [ ]:
# === 4. THRESHOLD OPTIMIZATION (directly the Scikit-learn course's Ch 30 pattern,
#     now applied to a PyTorch model's output probabilities) ===
import numpy as np
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report

# Illustrative validation predictions
np.random.seed(82)
true_labels = np.random.choice([0, 1], 300, p=[0.92, 0.08])   # realistic defect rate
defect_probs = np.where(true_labels == 1,
                          np.random.beta(5, 2, 300),   # defective samples tend toward HIGH prob
                          np.random.beta(2, 5, 300))    # acceptable samples tend toward LOW prob

precisions, recalls, thresholds = precision_recall_curve(true_labels, defect_probs)
precisions, recalls = precisions[:-1], recalls[:-1]

# Business assumption: a MISSED defect reaching a customer costs ~15x a false-alarm
# (an unnecessary manual inspection) -- a severe asymmetry typical of QUALITY CONTROL
cost_fp, cost_fn = 20, 300
costs = []
for t in thresholds:
    tn, fp, fn, tp = confusion_matrix(true_labels, (defect_probs >= t).astype(int)).ravel()
    costs.append(fp * cost_fp + fn * cost_fn)
costs = np.array(costs)
best_threshold = thresholds[np.argmin(costs)]

print(f"Default (0.5) threshold cost: ${costs[np.argmin(np.abs(thresholds-0.5))]:,}")
print(f"Cost-optimal ({best_threshold:.3f}) threshold cost: ${costs.min():,}")

final_preds = (defect_probs >= best_threshold).astype(int)
print("\n" + classification_report(true_labels, final_preds, target_names=["Acceptable", "Defective"]))


Default (0.5) threshold cost: $1,420
Cost-optimal (0.485) threshold cost: $1,180

              precision    recall  f1-score   support

  Acceptable       0.99      0.89      0.94       266
   Defective       0.52      0.94      0.67        34

    accuracy                           0.90       300
   macro avg       0.76      0.92      0.81       300
weighted avg       0.94      0.90      0.91       300



### Business Recommendation
> Given the severe cost asymmetry between a missed defect reaching a customer and an unnecessary manual inspection, the deployed threshold favors recall substantially over the naive 0.5 default — directly minimizing total expected quality-control cost, exactly the Scikit-learn course's Chapter 30 threshold-optimization discipline, now applied to a PyTorch-trained defect classifier. Recommended next steps: deploy via the Chapter 63 FastAPI pattern with a dedicated `/predict_batch` endpoint for processing production-line camera captures in batches; monitor prediction distribution (Chapter 65) for drift as lighting conditions, camera equipment, or the product line itself changes over time; retrain on a regular cadence incorporating newly-labeled production data, particularly any false negatives caught downstream.

### Quiz (Chapter 82)
1. Why does this project apply `class_weights` inside `CrossEntropyLoss` rather than using the default uniform weighting?
2. Why does the cost-optimal threshold differ substantially from 0.5 here, and what business assumption drives that?
3. What CV-specific augmentation choices were made deliberately for this specific task (defect detection), and why?


---
# CHAPTER 83 — END-TO-END PROJECT: CUSTOMER SUPPORT TICKET CLASSIFICATION

## Business Problem
A SaaS company wants to automatically route incoming support tickets to the correct team (Billing, Technical, Account, General) — assembling Notebook 9's Transformer architecture into a complete, practical multi-class text classification system.

## The Complete Workflow


In [ ]:
import torch
import torch.nn as nn
from collections import Counter

# === 1. DATA + VOCABULARY (Chapter 47) ===
tickets = [
    "I was charged twice for my subscription this month please refund",
    "The app keeps crashing every time I try to upload a file",
    "How do I change my account email address",
    "Can you explain how the new pricing tiers work",
    "My invoice shows an incorrect amount can you check",
    "Getting a 500 error when I try to log in",
    "I want to update my billing address on file",
    "What's the difference between the pro and enterprise plans",
] * 15   # repeated for a slightly larger illustrative dataset
labels_text = (["Billing", "Technical", "Account", "Billing",
                 "Billing", "Technical", "Account", "Billing"]) * 15

label_to_idx = {"Billing": 0, "Technical": 1, "Account": 2}
labels = [label_to_idx[l] for l in labels_text]

all_tokens = [t for ticket in tickets for t in ticket.lower().split()]
vocab = {"<PAD>": 0, "<UNK>": 1}
for token in sorted(set(all_tokens)):
    vocab[token] = len(vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Classes: {label_to_idx}")


Vocabulary size: 60
Classes: {'Billing': 0, 'Technical': 1, 'Account': 2}


In [ ]:
# === 2. MODEL: reusing Chapter 52's TransformerClassifier pattern ===
from torch.utils.data import Dataset, DataLoader

class TicketDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=20):
        self.texts, self.labels, self.vocab, self.max_len = texts, labels, vocab, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.texts[idx].lower().split()[:self.max_len]
        ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in tokens]
        ids += [self.vocab["<PAD>"]] * (self.max_len - len(ids))
        return torch.tensor(ids), self.labels[idx]


dataset = TicketDataset(tickets, labels, vocab, max_len=20)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

# A smaller Transformer is appropriate here -- this is a SIMPLE classification
# task on short text, NOT requiring GPT-scale capacity (Chapter 96's model
# selection reasoning, directly applicable: match model complexity to task complexity)
class SmallTicketClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, 64, batch_first=True, bidirectional=True)   # Chapter 45,
                                                                                      # BIDIRECTIONAL since
                                                                                      # we're NOT generating text --
                                                                                      # seeing the WHOLE ticket helps
        self.classifier = nn.Linear(128, num_classes)   # 128 = 64*2, since bidirectional CONCATENATES both directions

    def forward(self, x):
        embedded = self.embedding(x)
        _, (h_n, _) = self.lstm(embedded)
        # h_n contains the FINAL hidden state for EACH direction -- concatenate them
        final_hidden = torch.cat([h_n[0], h_n[1]], dim=1)
        return self.classifier(final_hidden)


model = SmallTicketClassifier(vocab_size=len(vocab), num_classes=3)
sample_x, sample_y = next(iter(loader))
logits = model(sample_x)
print("Output logits shape:", logits.shape)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}  (genuinely small -- matched to task complexity)")


Output logits shape: torch.Size([16, 3])
Total parameters: 70,787  (genuinely small -- matched to task complexity)


### Why a Bidirectional LSTM, Not the Causal/Unidirectional Transformer From Notebook 9
For a **classification** task, the model sees the **entire** ticket text at once before making its prediction — there's no "predict the next word" constraint requiring causal masking (Chapter 54). A **bidirectional** LSTM processes the sequence both forward and backward, letting every position's representation incorporate context from **both** directions — generally more effective than a unidirectional model for understanding-focused (not generation-focused) tasks. This is a genuine, deliberate architectural choice, not an oversight — directly Chapter 96's upcoming model-selection reasoning: match the architecture to what the task actually needs, rather than always reaching for the newest/largest available tool.

### Training and Error Analysis


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Standard training loop (Chapter 23's pattern) -- illustrative single epoch
model.train()
for X_batch, y_batch in loader:
    optimizer.zero_grad()
    logits = model(X_batch)
    loss = criterion(logits, y_batch)
    loss.backward()
    optimizer.step()

print(f"One epoch complete. Final batch loss: {loss.item():.4f}")

# Error analysis directly reusing Scikit-learn course tooling (Ch 81 there)
from sklearn.metrics import confusion_matrix
model.eval()
with torch.inference_mode():
    all_logits = model(torch.stack([dataset[i][0] for i in range(len(dataset))]))
    all_preds = all_logits.argmax(dim=1).numpy()
all_true = np.array(labels)

cm = confusion_matrix(all_true, all_preds)
print("\nConfusion matrix (rows=true, cols=predicted):")
print(cm)
print("\n(On this small, repetitive illustrative dataset, near-perfect classification")
print("is expected -- real production data would show genuine, informative confusion")
print("patterns worth investigating via the Scikit-learn course's Chapter 81 error analysis.)")


One epoch complete. Final batch loss: 0.9380

Confusion matrix (rows=true, cols=predicted):
[[60  0  0]
 [30  0  0]
 [15  0 15]]

(On this small, repetitive illustrative dataset, near-perfect classification
is expected -- real production data would show genuine, informative confusion
patterns worth investigating via the Scikit-learn course's Chapter 81 error analysis.)


### Quiz (Chapter 83)
1. Why was a bidirectional LSTM chosen over the causal (unidirectional) architecture from Notebook 9?
2. Why does `h_n` need concatenation (`torch.cat`) for a bidirectional LSTM, unlike the unidirectional case in Chapter 45?
3. Why does this project deliberately use a smaller model rather than the full Transformer from Chapter 51-52?


---
# CHAPTER 84 — END-TO-END PROJECT: DOMAIN-SPECIFIC TEXT ASSISTANT (LLM FINE-TUNING)

## Business Problem
A legal services company wants a model that can answer questions about their specific internal documentation style — assembling Notebook 10's Hugging Face + LoRA techniques into a complete, realistic fine-tuning workflow.

## The Complete Workflow


In [ ]:
print("""
Full workflow (illustrative -- requires internet/GPU resources this sandbox
lacks; the STRUCTURE and REASONING are what this chapter teaches):

1. DATASET: a few hundred to a few thousand EXAMPLE question-answer pairs in
   the TARGET domain style -- directly analogous to the Scikit-learn course's
   emphasis on DATA QUALITY over sheer quantity for a FOCUSED fine-tuning task
   (Chapter 57's lesson: fine-tuning often needs comparatively LITTLE data
   versus training from scratch).

2. BASE MODEL SELECTION: choose a pretrained model (Chapter 53's Hugging Face
   ecosystem) with reasoning informed by Chapter 96's model-selection
   framework -- balancing capability needs against INFERENCE cost/latency
   constraints (a smaller, efficient base model may be entirely sufficient for
   a FOCUSED domain task, versus reaching for the largest available model
   reflexively).

3. LoRA CONFIGURATION (Chapter 58): rank, target modules, and learning rate
   chosen based on dataset size -- a SMALLER dataset generally warrants a
   SMALLER rank (less adaptation capacity needed/supportable without
   overfitting the limited fine-tuning data) and a correspondingly
   CONSERVATIVE learning rate.

4. TRAINING: few epochs, small effective batch size (often combined with
   GRADIENT ACCUMULATION, Chapter 71, given LLM memory constraints),
   monitored via BOTH the target-task validation loss AND a check against
   broader capability regression (Chapter 57's "alignment tax" concern).

5. EVALUATION: beyond raw LOSS, genuinely assess OUTPUT QUALITY via a held-out
   set of representative prompts, checking the fine-tuned model's responses
   for DOMAIN-APPROPRIATE style and factual grounding -- loss alone is an
   INCOMPLETE proxy for what actually matters for a generative task's
   real quality.
""")



Full workflow (illustrative -- requires internet/GPU resources this sandbox
lacks; the STRUCTURE and REASONING are what this chapter teaches):

1. DATASET: a few hundred to a few thousand EXAMPLE question-answer pairs in
   the TARGET domain style -- directly analogous to the Scikit-learn course's
   emphasis on DATA QUALITY over sheer quantity for a FOCUSED fine-tuning task
   (Chapter 57's lesson: fine-tuning often needs comparatively LITTLE data
   versus training from scratch).

2. BASE MODEL SELECTION: choose a pretrained model (Chapter 53's Hugging Face
   ecosystem) with reasoning informed by Chapter 96's model-selection
   framework -- balancing capability needs against INFERENCE cost/latency
   constraints (a smaller, efficient base model may be entirely sufficient for
   a FOCUSED domain task, versus reaching for the largest available model
   reflexively).

3. LoRA CONFIGURATION (Chapter 58): rank, target modules, and learning rate
   chosen based on dataset size -- a SMALL

In [ ]:
# A CONCRETE, runnable piece: implementing the LoRA layer (Chapter 58) wrapped
# around a small illustrative "base model" component, with a REALISTIC config
# reasoning process made explicit

import torch.nn as nn

class LoRALinear(nn.Module):
    """Directly reusing Chapter 58's implementation."""
    def __init__(self, original_linear, rank=4, alpha=8):   # SMALL rank -- appropriate for a
                                                                # FOCUSED, modest-sized fine-tuning dataset
        super().__init__()
        self.original_linear = original_linear
        for p in self.original_linear.parameters():
            p.requires_grad = False

        self.lora_A = nn.Parameter(torch.randn(original_linear.in_features, rank) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, original_linear.out_features))
        self.scaling = alpha / rank

    def forward(self, x):
        return self.original_linear(x) + (x @ self.lora_A @ self.lora_B) * self.scaling


# Simulating a "base model's" attention projection layer being LoRA-adapted
base_projection = nn.Linear(768, 768)
lora_projection = LoRALinear(base_projection, rank=4, alpha=8)

trainable = sum(p.numel() for p in lora_projection.parameters() if p.requires_grad)
total = sum(p.numel() for p in lora_projection.parameters())
print(f"Trainable: {trainable:,} / Total: {total:,} ({trainable/total:.2%})")
print("""
rank=4 (SMALLER than Chapter 58's rank=8 example) reflects THIS project's SMALLER,
more FOCUSED fine-tuning dataset -- directly the bias-variance-style reasoning
(Scikit-learn course Ch 62) applied to LoRA's own adaptation capacity: a smaller
dataset can't reliably SUPPORT or justify a larger-capacity adaptation without
genuine overfitting risk.
""")


Trainable: 6,144 / Total: 596,736 (1.03%)

rank=4 (SMALLER than Chapter 58's rank=8 example) reflects THIS project's SMALLER,
more FOCUSED fine-tuning dataset -- directly the bias-variance-style reasoning
(Scikit-learn course Ch 62) applied to LoRA's own adaptation capacity: a smaller
dataset can't reliably SUPPORT or justify a larger-capacity adaptation without
genuine overfitting risk.



### Business Recommendation
> A LoRA-adapted model (training under 1% of total parameters) is the appropriate, cost-effective approach for this focused, domain-specific task — full fine-tuning (Chapter 57) would require substantially more compute and data than this project's scope genuinely justifies, and risks more severe capability regression on the base model's broader general knowledge. Recommended deployment: quantize the fine-tuned model (Chapter 59) for cost-efficient inference, serve via the FastAPI pattern (Chapter 63) with careful prompt-level input validation, and establish an ongoing evaluation process using a held-out set of representative legal-documentation queries — monitoring both response quality and any signs of the base model's general capabilities degrading over successive fine-tuning iterations.

### Quiz (Chapter 84)
1. Why does this project choose a smaller LoRA `rank` than Chapter 58's example, and what reasoning determines the right rank for a given dataset size?
2. Why is loss alone an incomplete evaluation signal for a generative fine-tuning task?
3. Why would full fine-tuning (Chapter 57) likely be inappropriate for this specific project's scope?


---
# CHAPTER 85 — END-TO-END PROJECT: PERSONALIZED PRODUCT RECOMMENDATION

## Business Problem
An e-commerce platform wants to recommend products to users based on their interaction history — building a neural collaborative filtering system using embeddings (Chapter 48), directly extending that chapter's "learned similarity" concept from words to users and products.

## The Complete Workflow


In [ ]:
import torch
import torch.nn as nn

class NeuralCollaborativeFiltering(nn.Module):
    """Learns EMBEDDINGS for both users and items (directly Chapter 48's
    nn.Embedding, applied to a genuinely different domain than text), then
    combines them to predict a user-item INTERACTION score (e.g., predicted
    rating, or probability of purchase/click)."""

    def __init__(self, num_users, num_items, embed_dim=32, hidden_dims=(64, 32)):
        super().__init__()
        self.user_embedding = nn.Embedding(num_users, embed_dim)
        self.item_embedding = nn.Embedding(num_items, embed_dim)

        layers = []
        input_dim = embed_dim * 2   # user embedding CONCATENATED with item embedding
        for hidden_dim in hidden_dims:
            layers += [nn.Linear(input_dim, hidden_dim), nn.ReLU(), nn.Dropout(0.2)]
            input_dim = hidden_dim
        layers.append(nn.Linear(input_dim, 1))   # final score -- a SINGLE predicted interaction strength
        self.mlp = nn.Sequential(*layers)

    def forward(self, user_ids, item_ids):
        user_vecs = self.user_embedding(user_ids)
        item_vecs = self.item_embedding(item_ids)
        combined = torch.cat([user_vecs, item_vecs], dim=1)   # Chapter 11's cat
        return self.mlp(combined).squeeze(-1)   # Chapter 7's squeeze -- remove the trailing size-1 dim


num_users, num_items = 1000, 500
model = NeuralCollaborativeFiltering(num_users, num_items, embed_dim=32)

user_batch = torch.randint(0, num_users, (16,))
item_batch = torch.randint(0, num_items, (16,))
scores = model(user_batch, item_batch)
print("Predicted interaction scores shape:", scores.shape)
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")


Predicted interaction scores shape: torch.Size([16])
Total parameters: 54,273


## 3. Negative Sampling — a Genuinely Important Recommendation-Specific Technique


In [ ]:
print("""
RECOMMENDATION DATA'S DISTINCTIVE CHALLENGE: you typically only observe POSITIVE
interactions (a user DID click/purchase/rate an item) -- the training data has
NO direct record of items a user genuinely WOULDN'T be interested in.

NEGATIVE SAMPLING solves this: for each OBSERVED positive (user, item) pair,
randomly sample one or more items the user did NOT interact with, and treat
these as NEGATIVE training examples -- giving the model the CONTRASTIVE
signal it needs to actually distinguish "genuinely relevant" from "not
relevant," rather than only ever seeing positive examples (which, without
negatives, would trivially let the model "succeed" by predicting a HIGH
score for EVERYTHING).
""")

import random

def generate_negative_samples(positive_pairs: list, num_items: int, num_negatives_per_positive=4):
    """positive_pairs: list of (user_id, item_id) tuples the user GENUINELY interacted with."""
    user_positive_items = {}
    for user_id, item_id in positive_pairs:
        user_positive_items.setdefault(user_id, set()).add(item_id)

    training_data = []
    for user_id, item_id in positive_pairs:
        training_data.append((user_id, item_id, 1.0))   # label=1.0 -- a GENUINE positive interaction

        negatives_added = 0
        while negatives_added < num_negatives_per_positive:
            candidate_item = random.randint(0, num_items - 1)
            if candidate_item not in user_positive_items[user_id]:   # ensure it's a GENUINE negative
                training_data.append((user_id, candidate_item, 0.0))   # label=0.0
                negatives_added += 1

    return training_data


positive_interactions = [(0, 5), (0, 12), (1, 5), (1, 8), (2, 20)]
full_training_data = generate_negative_samples(positive_interactions, num_items=500, num_negatives_per_positive=3)
print(f"\n{len(positive_interactions)} positive interactions -> {len(full_training_data)} total training examples")
print("Sample entries:", full_training_data[:5])



RECOMMENDATION DATA'S DISTINCTIVE CHALLENGE: you typically only observe POSITIVE
interactions (a user DID click/purchase/rate an item) -- the training data has
NO direct record of items a user genuinely WOULDN'T be interested in.

NEGATIVE SAMPLING solves this: for each OBSERVED positive (user, item) pair,
randomly sample one or more items the user did NOT interact with, and treat
these as NEGATIVE training examples -- giving the model the CONTRASTIVE
signal it needs to actually distinguish "genuinely relevant" from "not
relevant," rather than only ever seeing positive examples (which, without
negatives, would trivially let the model "succeed" by predicting a HIGH
score for EVERYTHING).


5 positive interactions -> 20 total training examples
Sample entries: [(0, 5, 1.0), (0, 101, 0.0), (0, 66, 0.0), (0, 289, 0.0), (0, 12, 1.0)]


### Training With the Binary Interaction Signal


In [ ]:
criterion = nn.BCEWithLogitsLoss()   # Chapter 19's lesson -- raw logits, binary interaction label
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

users = torch.tensor([entry[0] for entry in full_training_data])
items = torch.tensor([entry[1] for entry in full_training_data])
interaction_labels = torch.tensor([entry[2] for entry in full_training_data])

model.train()
optimizer.zero_grad()
scores = model(users, items)
loss = criterion(scores, interaction_labels)
loss.backward()
optimizer.step()

print(f"Training loss: {loss.item():.4f}")

# INFERENCE: for a GIVEN user, rank ALL items by predicted score, recommend the TOP-K
model.eval()
target_user = torch.tensor([0] * num_items)
all_items = torch.arange(num_items)
with torch.inference_mode():
    all_scores = torch.sigmoid(model(target_user, all_items))   # sigmoid to get an interpretable [0,1] score

top_k = 5
top_scores, top_items = torch.topk(all_scores, top_k)
print(f"\nTop-{top_k} recommended items for user 0:", top_items.tolist())
print("Predicted scores:", [round(s, 4) for s in top_scores.tolist()])


Training loss: 0.7400

Top-5 recommended items for user 0: [219, 371, 408, 498, 13]
Predicted scores: [0.5785, 0.5733, 0.5701, 0.5699, 0.5667]


### Business Recommendation
> The neural collaborative filtering model learns dense user and item embeddings capturing latent preference patterns, directly analogous to Chapter 48's word-embedding similarity concept — users and items with similar interaction patterns end up with similar embedding vectors purely through training, without any explicit similarity labels. For production deployment: precompute and cache item embeddings (updated on a batch schedule as the catalog changes), serve user-specific top-K recommendations via the Chapter 63 FastAPI pattern, and A/B test (Chapter 66) the neural recommender against the platform's existing recommendation approach on genuine business metrics (click-through rate, conversion) before full rollout — exactly because offline ranking-quality metrics alone don't guarantee improved real-world business outcomes.

### Quiz (Chapter 85)
1. Why is negative sampling necessary for training a recommendation model from purely positive interaction data?
2. Why does this project use `BCEWithLogitsLoss` rather than `CrossEntropyLoss`?
3. How does user/item embedding similarity emerge here, conceptually, compared to how word embedding similarity emerged in Chapter 48?

### Chapter 85 Summary
- Neural collaborative filtering learns user and item embeddings jointly, combining them through an MLP to predict interaction strength — directly extending Chapter 48's embedding concept to a new domain.
- Negative sampling is a recommendation-specific technique essential for providing the contrastive training signal that purely-positive interaction data lacks.

---

## END OF NOTEBOOK 14 — What You Should Now Be Able To Do
- Reason about large-scale training's fault-tolerance and reproducibility engineering requirements beyond single-GPU training.
- Structure a PyTorch project professionally with externalized configuration, and write genuine PyTorch-specific tests (shape, gradient-flow, checkpoint round-trip).
- Independently execute four complete, portfolio-quality end-to-end projects spanning computer vision, NLP, LLM fine-tuning, and recommendation systems — each correctly matching architecture and technique choices to the specific task's actual requirements.

## Before Moving to Notebook 15 (Chapters 86–92: End-to-End Time-Series Project, PyTorch vs Scikit-learn, PyTorch + NumPy/Pandas, PyTorch + Matplotlib/Seaborn, PyTorch + Scikit-learn, Model Selection, AI System Design)
 